In [1]:
"""
=============================================================================
ORANGE PROBLEM: VQE Simulation of the Schwinger Model
=============================================================================
Base paper: Melzer et al., arXiv:2504.20824 (2025)
Course: UE23EC342BB1 | Authors: Karthikeya Machiraju, Krishna Sujith

WHAT THIS CODE DOES:
  1. Builds the correct 4-qubit spin Hamiltonian W from scratch
  2. Implements the fermionic exchange ansatz (Uxy + Rz gates)
  3. Runs VQE for K = -14, 0, +10 (three phases)
  4. Compares against exact diagonalization
  5. Produces the full phase diagram and all report figures

HOW TO RUN:
  pip install numpy scipy matplotlib
  python schwinger_orange.py

LEARNING STRUCTURE:
  Read each section top to bottom. Every formula has a comment explaining
  WHY it looks the way it does, not just what it is.
=============================================================================
"""

import numpy as np
from scipy.linalg import eigh          # exact diagonalization
from scipy.optimize import minimize    # classical optimizer
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# =============================================================================
# SECTION 1: PAULI MATRICES AND TENSOR PRODUCTS
# =============================================================================
# These are the building blocks. Every operator in the Hamiltonian is built
# as a tensor product of these 2x2 matrices acting on individual qubits.

I2 = np.eye(2, dtype=complex)
X2 = np.array([[0, 1], [1, 0]], dtype=complex)
Y2 = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z2 = np.array([[1, 0], [0, -1]], dtype=complex)

def kron4(a, b, c, d):
    """
    Tensor product of 4 operators -> 16x16 matrix acting on 4 qubits.
    kron4(X, Z, X, I) means: X on qubit 0, Z on qubit 1, X on qubit 2, I on qubit 3.
    The ordering is qubit 0 = most significant bit (MSB).
    """
    return np.kron(np.kron(np.kron(a, b), c), d)


# =============================================================================
# SECTION 2: THE HAMILTONIAN
# =============================================================================
#
# We simulate N=2 lattice sites, F=2 fermion flavors -> 4 qubits.
# Qubit labeling: p = n*F + f
#   p=0: site n=0, flavor f=0
#   p=1: site n=0, flavor f=1
#   p=2: site n=1, flavor f=0
#   p=3: site n=1, flavor f=1
#
# The dimensionless Hamiltonian W (from paper Eq.5) has three parts:
#
# PART 1 - KINETIC (hopping):
#   For each neighboring site pair (n, n+1) and each flavor f,
#   the Jordan-Wigner transformed hopping is:
#       -x/2 * (XX + YY) with a Z-string between the two qubits
#   The Z-string encodes fermionic anticommutation statistics.
#   For flavor f=0: qubits 0 and 2, Z-string on qubit 1 -> X0 Z1 X2 I3
#   For flavor f=1: qubits 1 and 3, Z-string on qubit 2 -> I0 X1 Z2 X3
#
# PART 2 - CHEMICAL POTENTIAL:
#   nu_f/2 * (Z_{n,f} + I) for EACH site n, NO staggered sign.
#   WHY NO STAGGERED SIGN? The full term is (mu_f*(-1)^n + nu_f)/2*(Z+1).
#   The staggered (-1)^n applies ONLY to the mass term mu_f.
#   For massless fermions (mu=0 as in the paper), only nu_f remains,
#   and it applies uniformly to ALL sites.
#   CRITICAL: Getting this wrong reproduces wrong energies for K != 0.
#
# PART 3 - ELECTRIC FIELD (Gauss law):
#   After gauge field elimination, this becomes (Q_0)^2 for N=2.
#   Q_0 = (Z_0 + 1)/2 + (Z_1 + 1)/2
#   This counts the total charge at site n=0.

def build_hamiltonian(x, K):
    """
    Build the 16x16 dimensionless Hamiltonian W for N=2, F=2.

    Parameters
    ----------
    x : float
        Dimensionless coupling x = 1/(ag)^2. Use x=16 to match the paper.
    K : float
        Chemical potential parameter K = kappa_0/g - kappa_1/g.
        kappa_1/g = 0 throughout (paper convention).
        nu_0 = 2*sqrt(x)*K is the dimensionless chemical potential.

    Returns
    -------
    H : np.ndarray, shape (16,16), dtype complex
        The full Hamiltonian matrix in the 4-qubit computational basis.
    """
    nu0 = 2.0 * np.sqrt(x) * K   # dimensionless chemical potential for flavor 0
    H = np.zeros((16, 16), dtype=complex)

    # --- KINETIC TERM ---
    # Flavor 0: hops between qubit 0 (site 0) and qubit 2 (site 1)
    # The Z-string sits on qubit 1 (between them in Jordan-Wigner order)
    H += -x/2 * kron4(X2, Z2, X2, I2)   # X_0 Z_1 X_2 I_3
    H += -x/2 * kron4(Y2, Z2, Y2, I2)   # Y_0 Z_1 Y_2 I_3

    # Flavor 1: hops between qubit 1 (site 0) and qubit 3 (site 1)
    # The Z-string sits on qubit 2
    H += -x/2 * kron4(I2, X2, Z2, X2)   # I_0 X_1 Z_2 X_3
    H += -x/2 * kron4(I2, Y2, Z2, Y2)   # I_0 Y_1 Z_2 Y_3

    # --- CHEMICAL POTENTIAL TERM ---
    # nu0/2 * (Z + I) for flavor 0 at BOTH sites (no staggered sign on nu)
    H += nu0/2 * (kron4(Z2, I2, I2, I2) + np.eye(16))  # site n=0, flavor f=0
    H += nu0/2 * (kron4(I2, I2, Z2, I2) + np.eye(16))  # site n=1, flavor f=0
    # kappa_1 = 0 so no contribution from flavor 1

    # --- ELECTRIC FIELD TERM ---
    # Q_0 = (Z_0 + 1)/2 + (Z_1 + 1)/2 as a 16x16 matrix
    Q0 = (0.5 * (kron4(Z2, I2, I2, I2) + np.eye(16)) +
          0.5 * (kron4(I2, Z2, I2, I2) + np.eye(16)))
    H += Q0 @ Q0   # (Q_0)^2

    return H


# =============================================================================
# SECTION 3: THE PHYSICAL HILBERT SPACE (Qtot = 0 sector)
# =============================================================================
#
# Gauss's law requires Q_tot = 0. For N=2, F=2:
#   Q_tot = (Z_0 + Z_1 + Z_2 + Z_3) / 2
# Q_tot = 0 means: sum of all Z eigenvalues = 0,
# i.e., exactly 2 qubits in |0> and 2 qubits in |1>.
# That's C(4,2) = 6 states out of 16.
#
# These 6 states form three particle-number sectors:
#   N=(2,0): only |0101> -> ONE state (Energy is ALWAYS 1.0 regardless of K)
#   N=(1,1): |0011>, |0110>, |1001>, |1100> -> FOUR states
#   N=(0,2): only |1010> -> ONE state (Energy is ALWAYS 1.0 regardless of K)
#
# QUIZ: Why is N=(2,0) energy always 1.0?
# |0101>: Z values are [+1,-1,+1,-1]
# Chemical potential: nu0/2*(+1+1) + nu0/2*(+1+1) = 2*nu0 - 2*nu0 = 0 (cancels!)
# Electric field: Q0 = (1+1)/2 + (-1+1)/2 = 1, so (Q0)^2 = 1
# So E(|0101>) = 0 + 1 = 1 for ALL K. This is why Phase I has E=-223 at K=-14:
# the energy is NOT in the N=(2,0) sector energy, but in the N=(1,1) sector!
# Wait... let me re-examine. Actually E(N=(2,0)) at K=-14 is:
# nu0 = 2*4*(-14) = -112, E = 2*(-112) + 1 = -223. YES it does depend on K.
# The cancellation only happens when we sum Z values naively.
# The (Z+I)/2 operator gives 1 for |0> and 0 for |1>.
# For |0101>: N0 = 1+1=2, nu0 * N0 = -112*2 = -224 WAIT this doesnt work either.
# Let me be explicit: nu0/2*(Z0+I) for |0101>:
#   Z0|0101> = Z|0> tensor ... = +1|0101>, so <Z0> = +1
#   nu0/2*(Z0+I) contribution: nu0/2*(1+1) = nu0
# nu0/2*(Z2+I) for |0101>:
#   Z2|0101> -> Z acts on 3rd qubit which is |0>, so <Z2>=+1
#   nu0/2*(1+1) = nu0
# Total chemical: nu0 + nu0 = 2*nu0
# Electric: Q0 = (Z0+1)/2 + (Z1+1)/2 = 1 + 0 = 1, (Q0)^2 = 1
# So E(|0101>) = 2*nu0 + 1 = 2*2*sqrt(16)*K + 1 = 8K*4+1 = hmm
# nu0 = 2*sqrt(16)*K = 8*K
# E = 2*(8K) + 1 = 16K + 1. At K=-14: 16*(-14)+1 = -223. CORRECT!
#
# The key insight: Phases I and III are single product states whose
# energies are LINEAR in K. Phase II has entangled states with
# a more complex K dependence.

# The 6 Qtot=0 basis state indices (in decimal)
QTOT0_STATES = [3, 5, 6, 9, 10, 12]
#   3  = 0011  -> N=(1,1)
#   5  = 0101  -> N=(2,0)  <- initial state |psi_0>
#   6  = 0110  -> N=(1,1)
#   9  = 1001  -> N=(1,1)
#   10 = 1010  -> N=(0,2)
#   12 = 1100  -> N=(1,1)

def get_N0N1(state_idx):
    """
    Extract particle numbers N0, N1 from a computational basis state index.
    N_f = sum_n (Z_{n,f}+1)/2 = number of |0> qubits in flavor f positions.
    Flavor 0 lives on qubits 0 and 2. Flavor 1 lives on qubits 1 and 3.
    """
    bits = [(state_idx >> (3 - q)) & 1 for q in range(4)]  # q0=MSB
    N0 = sum(1 - bits[q] for q in [0, 2])  # |0>=particle for even sites
    N1 = sum(1 - bits[q] for q in [1, 3])
    return N0, N1

def project_to_sector(H_full, states):
    """Project a 16x16 matrix to a subspace defined by a list of state indices."""
    return np.array([[H_full[i, j] for j in states] for i in states])

def exact_ground_state(H_full, states=None):
    """
    Exact diagonalization. Returns (lowest eigenvalue, eigenvector as 16-dim vector).
    If states is given, projects to that subspace first.
    """
    if states is None:
        states = QTOT0_STATES
    Hsub = project_to_sector(H_full, states)
    vals, vecs = eigh(Hsub)
    # Reconstruct full 16-dim vector
    psi = np.zeros(16, dtype=complex)
    for idx, s in enumerate(states):
        psi[s] = vecs[idx, 0]
    return vals[0], psi

def particle_number_operator(flavor):
    """
    Build the 16x16 particle number operator for a given flavor.
    N_f = sum_{n=0}^{N-1} (Z_{n,f} + I) / 2
    """
    op = np.zeros((16, 16), dtype=complex)
    for n in range(2):
        qubit = n * 2 + flavor
        ops = [I2] * 4
        ops[qubit] = Z2
        op += 0.5 * (kron4(*ops) + np.eye(16))
    return op

N0_OP = particle_number_operator(0)
N1_OP = particle_number_operator(1)


# =============================================================================
# SECTION 4: THE ANSATZ (parameterized quantum circuit)
# =============================================================================
#
# The ansatz from paper Eq.(9) consists of two gate types:
#
# TYPE 1 - Uxy gate (fermionic exchange):
#   U^xy_ij(theta) = exp(-i*theta/2 * (Xi*Xj + Yi*Yj))
#   In matrix form on the 2-qubit subspace {|00>,|01>,|10>,|11>}:
#     |00> -> |00>
#     |01> -> cos(t/2)|01> - i*sin(t/2)|10>
#     |10> -> -i*sin(t/2)|01> + cos(t/2)|10>
#     |11> -> |11>
#   WHY THIS GATE? It preserves the total number of |1> qubits (total charge),
#   so the ansatz automatically stays in the Qtot=0 sector.
#
# TYPE 2 - Rz gate (single-qubit Z rotation):
#   Rz_i(theta) = exp(-i*theta/2 * Zi)
#   In matrix form: diag(e^{-i*theta/2}, e^{+i*theta/2})
#   This adds relative phases between |0> and |1> components.
#
# The full single-layer ansatz (7 parameters total):
#   U(theta) = [Rz_0 Rz_1 Rz_2 Rz_3] * Uxy_12(theta_1) * Uxy_23(theta_2) * Uxy_01(theta_0)
#
# WHY THIS ORDER?
#   1. Uxy_01 and Uxy_23 act on non-overlapping pairs -> can run in parallel
#   2. Uxy_12 creates CROSS-FLAVOR entanglement (mixes flavors 0 and 1)
#      This is the gate that allows transitions between N=(2,0) and N=(1,1) sectors
#   3. Rz gates add phases that help capture the correct ground state superposition
#
# EXPRESSIBILITY: For 4 qubits, Qtot=0 has dim=6.
# The ansatz has 7 parameters and can reach any state in this 6-dim space
# from |0101>. This means IT IS EXACT for this system size.

def Uxy_gate(theta):
    """4x4 fermionic exchange gate matrix."""
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    return np.array([
        [1,    0,      0,    0],
        [0,    c,  -1j*s,    0],
        [0, -1j*s,    c,    0],
        [0,    0,      0,    1]
    ], dtype=complex)

def Rz_gate(theta):
    """2x2 Z-rotation gate matrix."""
    return np.array([
        [np.exp(-1j * theta / 2), 0],
        [0, np.exp(+1j * theta / 2)]
    ], dtype=complex)

def apply_ansatz(theta, psi0):
    """
    Apply the single-layer ansatz U(theta) to initial state psi0.

    Parameters
    ----------
    theta : array of shape (7,)
        theta[0] = Uxy_01 angle
        theta[1] = Uxy_12 angle
        theta[2] = Uxy_23 angle
        theta[3:7] = Rz angles for qubits 0,1,2,3
    psi0 : array of shape (16,)
        Initial state vector.

    Returns
    -------
    psi : array of shape (16,)
        Output state vector.
    """
    I4 = np.eye(4, dtype=complex)

    # Embed 4x4 Uxy gate into 16x16 space
    # Uxy_01 acts on qubits 0,1 -> tensor with I on qubits 2,3
    U01 = np.kron(Uxy_gate(theta[0]), I4)
    # Uxy_23 acts on qubits 2,3 -> tensor I on qubits 0,1
    U23 = np.kron(I4, Uxy_gate(theta[2]))
    # Uxy_12 acts on qubits 1,2 -> I on qubit 0, Uxy on 1,2, I on qubit 3
    U12 = np.kron(np.kron(I2, Uxy_gate(theta[1])), I2)

    # Build combined Rz rotation on all 4 qubits
    Rz_all = np.kron(
        np.kron(np.kron(Rz_gate(theta[3]), Rz_gate(theta[4])),
                Rz_gate(theta[5])),
        Rz_gate(theta[6])
    )

    # Apply gates in order: Uxy_01, Uxy_23, then Uxy_12, then Rz
    psi = U01 @ psi0
    psi = U23 @ psi
    psi = U12 @ psi
    psi = Rz_all @ psi
    return psi

def energy_expectation(theta, H, psi0):
    """
    Cost function for VQE: <psi(theta)|H|psi(theta)>.
    This is the number the optimizer tries to minimize.
    """
    psi = apply_ansatz(theta, psi0)
    return float(np.real(psi.conj() @ H @ psi))


# =============================================================================
# SECTION 5: RUN VQE
# =============================================================================
#
# The optimization strategy:
#   1. Random initialization (multiple restarts to avoid local minima)
#   2. COBYLA optimizer: derivative-free, handles noisy cost functions
#   3. Report final energy, error, and particle numbers
#
# In the paper they use SPSA (stochastic gradient), which is noise-robust
# for real quantum hardware. For our noiseless classical simulation,
# COBYLA is more efficient.

def run_vqe(H, psi0, n_restarts=20, seed=42):
    """
    Run VQE with multiple random restarts and return the best result.

    Returns
    -------
    best_energy : float
    best_theta : array of shape (7,)
    """
    best_energy = np.inf
    best_theta = None
    rng = np.random.default_rng(seed)

    for restart in range(n_restarts):
        # Random initial angles in [-pi, pi]
        theta_init = rng.uniform(-np.pi, np.pi, 7)
        result = minimize(
            energy_expectation,
            theta_init,
            args=(H, psi0),
            method='COBYLA',
            options={'maxiter': 5000, 'rhobeg': 0.5}
        )
        if result.fun < best_energy:
            best_energy = result.fun
            best_theta = result.x

    return best_energy, best_theta

def run_vqe_with_trace(H, psi0, n_iters=150, seed=7):
    """
    Run VQE using gradient descent and record the energy at each step.
    This is for plotting convergence curves.
    Uses parameter-shift gradient estimation (standard in quantum computing).
    """
    rng = np.random.default_rng(seed)
    theta = rng.uniform(-np.pi, np.pi, 7)
    trace = [energy_expectation(theta, H, psi0)]
    eps = 1e-3

    for it in range(n_iters):
        # Compute gradient via finite differences
        grad = np.zeros(7)
        for j in range(7):
            theta_plus = theta.copy();  theta_plus[j] += eps
            theta_minus = theta.copy(); theta_minus[j] -= eps
            grad[j] = (energy_expectation(theta_plus, H, psi0) -
                       energy_expectation(theta_minus, H, psi0)) / (2 * eps)

        # Decaying learning rate + small noise (mimics SPSA)
        lr = 0.3 / (1 + 0.04 * it)
        noise = rng.standard_normal(7) * 0.01
        theta -= lr * grad + noise

        trace.append(energy_expectation(theta, H, psi0))

    return trace, theta


# =============================================================================
# SECTION 6: MAIN SIMULATION
# =============================================================================

def main():
    print("=" * 65)
    print("SCHWINGER MODEL VQE SIMULATION")
    print("N=2 sites, F=2 flavors, x=16, massless (m=0), kappa1=0")
    print("=" * 65)

    x = 16.0   # dimensionless coupling (reproduces paper Table I)

    # ---- Initial state: |0101> (charge-neutral, N0=2, N1=0) ----
    psi0 = np.zeros(16, dtype=complex)
    psi0[0b0101] = 1.0   # binary 0101 = decimal 5

    # ---- Verify Hamiltonian reproduces paper Table I ----
    print("\n[Step 1] Verifying Hamiltonian against paper Table I:")
    print(f"{'K':>6} | {'W_sim':>12} | {'W_paper':>10} | {'Match':>6}")
    print("-" * 42)
    for K, paper_val in [(-14, -223.0), (0, -30.7), (10, 1.0)]:
        H = build_hamiltonian(x, K)
        E, _ = exact_ground_state(H)
        match = "OK" if abs(E - paper_val) < 0.5 else "FAIL"
        print(f"K={K:>4} | {E:>12.4f} | {paper_val:>10.1f} | {match:>6}")

    # ---- Compute energies by sector for phase diagram ----
    print("\n[Step 2] Computing phase diagram (energies by sector)...")
    K_sweep = np.linspace(-16, 16, 200)
    sectors = {}
    for s in QTOT0_STATES:
        key = get_N0N1(s)
        sectors.setdefault(key, []).append(s)

    E_by_sector = {(2,0): [], (1,1): [], (0,2): []}
    E_global    = []
    N0_global   = []

    for K in K_sweep:
        H = build_hamiltonian(x, float(K))
        energies = {}
        for (n0, n1), states in sectors.items():
            Hsub = project_to_sector(H, states)
            energies[(n0, n1)] = float(eigh(Hsub)[0][0])
        for key in E_by_sector:
            E_by_sector[key].append(energies[key])
        E_global.append(min(energies.values()))
        # Track <N0> in the global ground state
        _, psi_gs = exact_ground_state(H)
        N0_global.append(float(np.real(psi_gs.conj() @ N0_OP @ psi_gs)))

    # Find phase boundaries (level crossings)
    E20 = np.array(E_by_sector[(2,0)])
    E11 = np.array(E_by_sector[(1,1)])
    E02 = np.array(E_by_sector[(0,2)])

    print("  Phase boundaries (level crossings):")
    for i in range(1, len(K_sweep)):
        if (E20[i-1]-E11[i-1]) * (E20[i]-E11[i]) < 0:
            Kc = K_sweep[i-1] + (K_sweep[i]-K_sweep[i-1]) * \
                 (E20[i-1]-E11[i-1]) / (E20[i-1]-E11[i-1]-E20[i]+E11[i])
            print(f"    (2,0) <-> (1,1): K_crit = {Kc:.4f}  [paper: -3.96]")
        if (E11[i-1]-E02[i-1]) * (E11[i]-E02[i]) < 0:
            Kc = K_sweep[i-1] + (K_sweep[i]-K_sweep[i-1]) * \
                 (E11[i-1]-E02[i-1]) / (E11[i-1]-E02[i-1]-E11[i]+E02[i])
            print(f"    (1,1) <-> (0,2): K_crit = {Kc:.4f}  [paper: +3.96]")

    # ---- VQE for the three paper phases ----
    print("\n[Step 3] Running VQE for K = -14, 0, +10...")
    print(f"\n{'Phase':>12} | {'K':>5} | {'E_exact':>12} | {'E_VQE':>12} | "
          f"{'Rel.Err%':>10} | {'N0':>5} | {'N1':>5}")
    print("-" * 72)

    vqe_results = {}
    conv_traces = {}

    for K, phase_label in [(-14.0, "I (2,0)"), (0.0, "II (1,1)"), (10.0, "III (0,2)")]:
        H = build_hamiltonian(x, K)
        E_exact, _ = exact_ground_state(H)

        # VQE optimization (multiple restarts)
        E_vqe, theta_opt = run_vqe(H, psi0, n_restarts=20, seed=42)

        # Measure particle numbers from optimized state
        psi_opt = apply_ansatz(theta_opt, psi0)
        N0_val = float(np.real(psi_opt.conj() @ N0_OP @ psi_opt))
        N1_val = float(np.real(psi_opt.conj() @ N1_OP @ psi_opt))

        rel_err = abs(E_vqe - E_exact) / abs(E_exact) * 100

        print(f"Phase {phase_label:>7} | K={K:>4.0f} | {E_exact:>12.4f} | "
              f"{E_vqe:>12.4f} | {rel_err:>10.4f} | {N0_val:>5.2f} | {N1_val:>5.2f}")

        # Convergence trace for plotting
        trace, _ = run_vqe_with_trace(H, psi0, n_iters=150, seed=7)

        vqe_results[K] = {
            'E_exact': E_exact, 'E_vqe': E_vqe,
            'rel_err_pct': rel_err, 'N0': N0_val, 'N1': N1_val,
            'phase': phase_label
        }
        conv_traces[K] = {'trace': trace, 'E_exact': E_exact}

    # ---- GENERATE ALL FIGURES ----
    print("\n[Step 4] Generating figures...")
    generate_figures(K_sweep, E20, E11, E02, E_global, N0_global,
                     vqe_results, conv_traces)
    print("  Figures saved: fig1_phase_diagram.png, fig2_convergence.png,")
    print("                 fig3_accuracy.png, fig4_particle_number.png")


# =============================================================================
# SECTION 7: FIGURE GENERATION
# =============================================================================

def generate_figures(K_sweep, E20, E11, E02, E_global, N0_global,
                     vqe_results, conv_traces):

    plt.rcParams.update({
        'font.family': 'serif', 'font.size': 11,
        'axes.labelsize': 12, 'axes.titlesize': 11,
        'legend.fontsize': 9, 'figure.facecolor': 'white'
    })

    # ---- Figure 1: Phase diagram ----
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Energy landscape
    ax1.plot(K_sweep, E20, 'b-',  lw=2.5, label=r'$\mathcal{N}=(2,0)$ sector')
    ax1.plot(K_sweep, E11, 'r-',  lw=2.5, label=r'$\mathcal{N}=(1,1)$ sector')
    ax1.plot(K_sweep, E02, 'g--', lw=2,   label=r'$\mathcal{N}=(0,2)$ sector')

    for K_c in [-3.9456, 3.9456]:
        ax1.axvline(K_c, color='gray', ls=':', lw=1.5, alpha=0.8)

    # VQE points as markers
    vqe_styles = {-14.0: ('b','o'), 0.0: ('r','s'), 10.0: ('g','^')}
    for K, res in vqe_results.items():
        c, m = vqe_styles[K]
        ax1.scatter([K], [res['E_vqe']], color=c, s=130, zorder=6,
                    marker=m, edgecolors='k', lw=0.8)

    ax1.axvspan(-16, -3.9456, alpha=0.05, color='blue')
    ax1.axvspan(-3.9456, 3.9456, alpha=0.05, color='red')
    ax1.axvspan(3.9456, 16, alpha=0.05, color='green')
    ax1.text(-10, 30, 'Phase I\n$\\mathcal{N}=(2,0)$',
             ha='center', fontsize=10, color='#1f77b4')
    ax1.text(0, 30, 'Phase II\n$\\mathcal{N}=(1,1)$',
             ha='center', fontsize=10, color='#d62728')
    ax1.text(10, 30, 'Phase III\n$\\mathcal{N}=(0,2)$',
             ha='center', fontsize=10, color='#2ca02c')
    ax1.set_xlabel(r'$K = \kappa_0/g$'); ax1.set_ylabel(r'$\langle W \rangle$')
    ax1.set_title('(a) Phase diagram: sector energies vs K')
    ax1.legend(loc='upper right'); ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-16, 16); ax1.set_ylim(-280, 70)

    # N0 expectation value
    ax2.plot(K_sweep, N0_global, 'k-', lw=2.5)
    for K_c in [-3.9456, 3.9456]:
        ax2.axvline(K_c, color='gray', ls=':', lw=1.5, alpha=0.8)
    for y in [0, 1, 2]:
        ax2.axhline(y, color='gray', ls='--', lw=0.8, alpha=0.5)
    for K, res in vqe_results.items():
        c, m = vqe_styles[K]
        ax2.scatter([K], [res['N0']], color=c, s=130, zorder=6,
                    marker=m, edgecolors='k', lw=0.8)
    ax2.axvspan(-16, -3.9456, alpha=0.05, color='blue')
    ax2.axvspan(-3.9456, 3.9456, alpha=0.05, color='red')
    ax2.axvspan(3.9456, 16, alpha=0.05, color='green')
    ax2.set_xlabel(r'$K = \kappa_0/g$')
    ax2.set_ylabel(r'$\langle N_0 \rangle$')
    ax2.set_title(r'(b) Flavor-0 particle number $\langle N_0 \rangle$ vs K')
    ax2.set_xlim(-16, 16); ax2.set_ylim(-0.2, 2.4)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('fig1_phase_diagram.png', dpi=200, bbox_inches='tight')
    plt.close()

    # ---- Figure 2: VQE convergence ----
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    colors = {-14.0: '#1f77b4', 0.0: '#d62728', 10.0: '#2ca02c'}
    titles = {-14.0: r'(a) $K=-14$: Phase I, $\mathcal{N}=(2,0)$',
               0.0:  r'(b) $K=0$: Phase II, $\mathcal{N}=(1,1)$',
              10.0:  r'(c) $K=+10$: Phase III, $\mathcal{N}=(0,2)$'}

    for ax, (K, ct) in zip(axes, conv_traces.items()):
        iters = range(len(ct['trace']))
        ax.plot(iters, ct['trace'], color=colors[K], lw=2, label='VQE gradient descent')
        ax.axhline(ct['E_exact'], color='k', ls='--', lw=1.5,
                   label=f"Exact: {ct['E_exact']:.3f}")
        ax.set_xlabel('Iteration'); ax.set_ylabel(r'$\langle W(\theta) \rangle$')
        ax.set_title(titles[K]); ax.legend(); ax.grid(True, alpha=0.3)
        res = vqe_results[K]
        ax.text(0.97, 0.05, f"VQE: {res['E_vqe']:.4f}\nErr: {res['rel_err_pct']:.4f}%",
                transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.9))

    plt.tight_layout()
    plt.savefig('fig2_convergence.png', dpi=200, bbox_inches='tight')
    plt.close()

    # ---- Figure 3: Accuracy comparison ----
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    K_labels = ['K = -14', 'K = 0', 'K = +10']
    K_keys   = [-14.0, 0.0, 10.0]
    E_ex  = [vqe_results[k]['E_exact']  for k in K_keys]
    E_vqe = [vqe_results[k]['E_vqe']    for k in K_keys]
    errs  = [vqe_results[k]['rel_err_pct'] for k in K_keys]

    x_bar = np.arange(3); w = 0.35
    bars1 = ax1.bar(x_bar - w/2, np.abs(E_ex),  w, label='Exact ED',
                    color='steelblue', edgecolor='k', lw=0.5)
    bars2 = ax1.bar(x_bar + w/2, np.abs(E_vqe), w, label='VQE',
                    color='salmon', edgecolor='k', lw=0.5, alpha=0.85)
    for i, v in enumerate(np.abs(E_ex)):
        ax1.text(i - w/2, v + 4, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
    for i, v in enumerate(np.abs(E_vqe)):
        ax1.text(i + w/2, v + 4, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
    ax1.set_xticks(x_bar); ax1.set_xticklabels(K_labels)
    ax1.set_ylabel(r'$|\langle W \rangle|$')
    ax1.set_title('(a) Exact ED vs VQE energy')
    ax1.legend(); ax1.grid(True, axis='y', alpha=0.3)

    bar_colors = ['steelblue', 'red', 'green']
    ax2.bar(x_bar, errs, color=bar_colors, edgecolor='k', lw=0.5)
    for i, v in enumerate(errs):
        ax2.text(i, v + 0.001, f'{v:.4f}%', ha='center', va='bottom', fontsize=10)
    ax2.set_xticks(x_bar); ax2.set_xticklabels(K_labels)
    ax2.set_ylabel('Relative error (%)'); ax2.set_title('(b) VQE relative error')
    ax2.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('fig3_accuracy.png', dpi=200, bbox_inches='tight')
    plt.close()

    # ---- Figure 4: Particle numbers ----
    fig, ax = plt.subplots(figsize=(8, 5))
    x_bar = np.arange(3); w = 0.3
    N0_vals = [vqe_results[k]['N0'] for k in K_keys]
    N1_vals = [vqe_results[k]['N1'] for k in K_keys]

    ax.bar(x_bar - w/2, N0_vals, w, label=r'$\langle N_0 \rangle$ (VQE)',
           color='cornflowerblue', edgecolor='k', lw=0.5)
    ax.bar(x_bar + w/2, N1_vals, w, label=r'$\langle N_1 \rangle$ (VQE)',
           color='coral', edgecolor='k', lw=0.5)

    # Paper hardware values for comparison
    paper_N0 = [1.73, 0.97, 0.17]
    ax.scatter(x_bar - w/2, paper_N0, color='navy', s=120, marker='*',
               zorder=5, label='Hardware (Melzer et al.)')

    for y in [0, 1, 2]:
        ax.axhline(y, color='gray', ls='--', lw=0.8, alpha=0.4)
    ax.set_xticks(x_bar); ax.set_xticklabels(K_labels, fontsize=12)
    ax.set_ylabel('Particle number'); ax.set_ylim(-0.2, 2.5)
    ax.set_title('Particle number measurements by phase')
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('fig4_particle_number.png', dpi=200, bbox_inches='tight')
    plt.close()


# =============================================================================
# SECTION 8: BONUS - VERIFY YOUR UNDERSTANDING
# =============================================================================
# Uncomment these to deepen your understanding.

def check_hamiltonian_properties():
    """
    Sanity checks every physics student should run on their Hamiltonian.
    """
    x = 16.0
    for K in [-14.0, 0.0, 10.0]:
        H = build_hamiltonian(x, K)
        print(f"\nK={K}: Hamiltonian checks")
        print(f"  Hermitian: {np.allclose(H, H.conj().T)}")
        print(f"  Real eigenvalues: ", end="")
        vals = np.linalg.eigvalsh(H)
        print(np.allclose(vals, np.real(vals)))
        # Check [H, N0] = 0 (particle number conservation)
        comm = H @ N0_OP - N0_OP @ H
        print(f"  [H, N0] = 0: {np.allclose(comm, 0, atol=1e-10)}")

def check_ansatz_conservation():
    """
    Verify that the ansatz preserves Qtot = 0.
    Starting from |0101> (Qtot=0), all output states should also have Qtot=0.
    """
    psi0 = np.zeros(16, dtype=complex); psi0[5] = 1.0
    Qtot_op = sum(0.5 * (kron4(*([I2 if q != qubit else Z2 for q in range(4)])) + np.eye(16))
                  for qubit in range(4))
    # Correct: Qtot = (Z0+Z1+Z2+Z3)/2
    Qtot_op2 = (kron4(Z2,I2,I2,I2) + kron4(I2,Z2,I2,I2) +
                kron4(I2,I2,Z2,I2) + kron4(I2,I2,I2,Z2)) / 2

    rng = np.random.default_rng(0)
    print("\nAnsatz conserves Qtot=0:")
    for trial in range(5):
        theta = rng.uniform(-np.pi, np.pi, 7)
        psi_out = apply_ansatz(theta, psi0)
        Qtot = float(np.real(psi_out.conj() @ Qtot_op2 @ psi_out))
        print(f"  Trial {trial}: <Qtot> = {Qtot:.10f}  (should be 0.0)")


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    main()

    # Uncomment to run verification checks:
    # print("\n" + "="*65)
    # print("VERIFICATION CHECKS")
    # print("="*65)
    # check_hamiltonian_properties()
    # check_ansatz_conservation()

SCHWINGER MODEL VQE SIMULATION
N=2 sites, F=2 flavors, x=16, massless (m=0), kappa1=0

[Step 1] Verifying Hamiltonian against paper Table I:
     K |        W_sim |    W_paper |  Match
------------------------------------------
K= -14 |    -223.0000 |     -223.0 |     OK
K=   0 |     -30.5644 |      -30.7 |     OK
K=  10 |       1.0000 |        1.0 |     OK

[Step 2] Computing phase diagram (energies by sector)...
  Phase boundaries (level crossings):
    (2,0) <-> (1,1): K_crit = -3.9456  [paper: -3.96]
    (1,1) <-> (0,2): K_crit = 3.9456  [paper: +3.96]

[Step 3] Running VQE for K = -14, 0, +10...

       Phase |     K |      E_exact |        E_VQE |   Rel.Err% |    N0 |    N1
------------------------------------------------------------------------
Phase I (2,0) | K= -14 |    -223.0000 |    -223.0000 |     0.0000 |  2.00 |  0.00
Phase II (1,1) | K=   0 |     -30.5644 |     -30.5606 |     0.0127 |  1.00 |  1.00
Phase III (0,2) | K=  10 |       1.0000 |       1.0000 |     0.0000 |  0.

In [1]:
"""
=============================================================================
VQE Study of the Multi-Flavor Schwinger Model
For conference publication (IEEE QCE 2026 target)

Authors: Karthikeya Machiraju, Krishna Sujith
Base paper: Melzer et al., arXiv:2504.20824 (2025)

WHAT THIS CODE STUDIES:
  1. Classical statevector VQE  -- exact, what we built in Orange
  2. Quantum circuit VQE with shot noise  -- NEW: what a real QC does
  3. Comparison: statevector vs shot noise vs hardware (paper Table I)
  4. Entanglement entropy across phases  -- NEW observable
  5. VQE scaling N=2 to 5 lattice sites  -- NEW, connects to Jackfruit

HOW TO RUN:
  pip install numpy scipy matplotlib qiskit qiskit-aer
  python schwinger_study.py

LEARNING APPROACH:
  Each section has detailed comments explaining the physics AND the code.
  Read them. Don't skip. This is your PhD foundation.
=============================================================================
"""

import numpy as np
from scipy.linalg import eigh
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# PART 0: CORE BUILDING BLOCKS
# Everything else depends on these. Understand them first.
# ============================================================================

# The four Pauli matrices. Every quantum operator on qubits is built from these.
I2 = np.eye(2, dtype=complex)
X2 = np.array([[0, 1], [1, 0]], dtype=complex)
Y2 = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z2 = np.array([[1, 0], [0, -1]], dtype=complex)

def kron_list(ops):
    """Tensor product of a list of operators: ops[0] ⊗ ops[1] ⊗ ... ⊗ ops[n-1]"""
    result = ops[0]
    for op in ops[1:]:
        result = np.kron(result, op)
    return result

def single_qubit_op(gate, qubit, n_qubits):
    """
    Embed a single-qubit gate into a n_qubits-qubit space.
    Acts as 'gate' on 'qubit', identity on everything else.
    Qubit 0 = most significant bit (leftmost in |q0 q1 q2 q3>).
    """
    ops = [I2] * n_qubits
    ops[qubit] = gate
    return kron_list(ops)

def two_qubit_op(gate1, q1, gate2, q2, n_qubits, z_string_between=None):
    """Build an operator with gates on two qubits and optional Z-string in between."""
    ops = [I2] * n_qubits
    ops[q1] = gate1
    ops[q2] = gate2
    if z_string_between is not None:
        for qz in z_string_between:
            ops[qz] = Z2
    return kron_list(ops)


# ============================================================================
# PART 1: THE HAMILTONIAN
# This is the heart of the whole simulation.
# ============================================================================

def build_hamiltonian(x, K, N=2, F=2):
    """
    Build the dimensionless W Hamiltonian for N lattice sites, F flavors.

    PHYSICAL MEANING OF PARAMETERS:
      x = 1/(ag)^2  -- dimensionless coupling. Large x = strong kinetic term.
                       The paper uses x=16, which means kinetic dominates at K=0.
      K = kappa_0/g  -- chemical potential for flavor 0 (in units of coupling g)
                        kappa_1 = 0 throughout (paper convention).
      N = lattice sites (paper uses N=2)
      F = fermion flavors (paper uses F=2)

    HAMILTONIAN STRUCTURE (three terms):

    Term 1 - KINETIC:
      -x/2 * (XX + YY) with Z-string between the two hopping qubits.
      The Z-string comes from the Jordan-Wigner transformation.
      Physically: fermion hops between neighboring sites, dragging a
      "string of phase" through all intermediate sites to maintain
      fermionic anticommutation statistics.

    Term 2 - CHEMICAL POTENTIAL:
      nu_0/2 * (Z_{n,f=0} + I) for BOTH sites n=0 and n=1.
      CRITICAL: NO staggered (-1)^n factor on nu. The staggered sign is
      only on the mass term mu. For massless fermions (mu=0 as in the paper),
      the chemical potential applies uniformly to all sites.
      nu_0 = 2*sqrt(x)*K is the dimensionless chemical potential.

    Term 3 - ELECTRIC FIELD (Gauss law enforced):
      sum_{n=0}^{N-2} (sum_{k=0}^{n} Q_k)^2
      This is the energy cost of electric flux, after gauge fields
      have been completely eliminated via Gauss's law.
      For N=2: only (Q_0)^2 where Q_0 = (Z_0+I)/2 + (Z_1+I)/2.

    Returns: H as a (2^(NF)) x (2^(NF)) complex matrix
    """
    nu0 = 2.0 * np.sqrt(x) * K
    n_qubits = N * F
    dim = 2 ** n_qubits
    H = np.zeros((dim, dim), dtype=complex)

    # --- Term 1: Kinetic ---
    for n in range(N - 1):           # loop over neighboring site pairs
        for f in range(F):           # loop over flavors
            p1 = n * F + f           # qubit for site n, flavor f
            p2 = (n + 1) * F + f     # qubit for site n+1, flavor f
            # Z-string: all qubits strictly between p1 and p2
            z_string = list(range(p1 + 1, p2))
            H += -x/2 * two_qubit_op(X2, p1, X2, p2, n_qubits, z_string)
            H += -x/2 * two_qubit_op(Y2, p1, Y2, p2, n_qubits, z_string)

    # --- Term 2: Chemical potential (flavor 0 only, nu_1=0) ---
    for n in range(N):
        qubit = n * F + 0            # qubit for site n, flavor 0
        Z_q = single_qubit_op(Z2, qubit, n_qubits)
        H += nu0 / 2 * (Z_q + np.eye(dim))

    # --- Term 3: Electric field ---
    for n in range(N - 1):
        # Build cumulative charge: sum_{k=0}^{n} Q_k
        cumQ = np.zeros((dim, dim), dtype=complex)
        for k in range(n + 1):
            for f in range(F):
                qubit = k * F + f
                Z_q = single_qubit_op(Z2, qubit, n_qubits)
                cumQ += 0.5 * (Z_q + np.eye(dim))
            # Subtract staggered offset: F/2 * (1 - (-1)^k)
            cumQ -= F / 2.0 * (1 - (-1)**k) * np.eye(dim)
        H += cumQ @ cumQ

    return H


def get_qtot0_basis(n_qubits, F=2, N=2):
    """
    Find all computational basis states in the Q_tot = 0 sector.

    Q_tot = 0 means: sum of all Z eigenvalues = 0.
    Since Z|0> = +1 and Z|1> = -1, this means equal numbers of |0> and |1> qubits.
    For 4 qubits: exactly 2 must be |0> and 2 must be |1>. That's C(4,2) = 6 states.

    For N sites, F flavors: C(NF, NF/2) states if we require Q_tot=0 with
    equal charges. (This is the charge-neutral sector.)
    """
    basis = []
    dim = 2 ** n_qubits
    for s in range(dim):
        bits = [(s >> (n_qubits - 1 - q)) & 1 for q in range(n_qubits)]
        Z_sum = sum(1 - 2 * b for b in bits)
        if Z_sum == 0:
            basis.append(s)
    return basis


def exact_diagonalize(H, basis):
    """
    Exact diagonalization restricted to the given subspace (basis states).
    Returns all eigenvalues and the full ground state vector.
    """
    Hsub = np.array([[H[i, j] for j in basis] for i in basis])
    vals, vecs = eigh(Hsub)
    # Lift eigenvector back into full Hilbert space
    dim = H.shape[0]
    psi_gs = np.zeros(dim, dtype=complex)
    for idx, state in enumerate(basis):
        psi_gs[state] = vecs[idx, 0]
    return vals, psi_gs


def particle_number_op(flavor, N=2, F=2):
    """
    N_f = sum_{n=0}^{N-1} (Z_{n,f} + I) / 2
    Counts the number of particles of flavor f.
    Convention: |0> = particle (Z=+1 gives (1+1)/2=1), |1> = hole.
    """
    n_qubits = N * F
    dim = 2 ** n_qubits
    op = np.zeros((dim, dim), dtype=complex)
    for n in range(N):
        qubit = n * F + flavor
        Z_q = single_qubit_op(Z2, qubit, n_qubits)
        op += 0.5 * (Z_q + np.eye(dim))
    return op


# ============================================================================
# PART 2: THE ANSATZ
# The parameterized quantum circuit that VQE optimizes.
# ============================================================================

def Uxy_matrix(theta):
    """
    Fermionic exchange gate: U^xy(theta) = exp(-i*theta/2*(XX+YY))

    Matrix form in the 2-qubit basis {|00>, |01>, |10>, |11>}:
      |00> -> |00>                       (unchanged)
      |01> -> cos(t/2)|01> - i*sin(t/2)|10>  (coherent mixing)
      |10> -> -i*sin(t/2)|01> + cos(t/2)|10> (coherent mixing)
      |11> -> |11>                       (unchanged)

    This gate conserves the total number of |1> qubits (excitations).
    That's WHY it keeps us in the Q_tot=0 sector -- it can never
    change the total charge, only redistribute it between qubits.

    At theta=0: identity.
    At theta=pi: full SWAP of |01> and |10> (with phase).
    """
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    return np.array([
        [1,     0,      0,  0],
        [0,     c,  -1j*s,  0],
        [0, -1j*s,      c,  0],
        [0,     0,      0,  1]
    ], dtype=complex)


def Rz_matrix(theta):
    """
    Z-rotation: R_z(theta) = exp(-i*theta/2 * Z)
    Adds a relative phase between |0> and |1>.
    R_z(theta)|0> = e^{-i*theta/2}|0>
    R_z(theta)|1> = e^{+i*theta/2}|1>
    """
    return np.array([
        [np.exp(-1j * theta / 2), 0],
        [0, np.exp(+1j * theta / 2)]
    ], dtype=complex)


def embed_Uxy(theta, q1, q2, n_qubits):
    """Embed the 4x4 Uxy gate acting on qubits q1,q2 into the full space."""
    dim = 2 ** n_qubits
    U = np.eye(dim, dtype=complex)
    Uxy = Uxy_matrix(theta)
    # Build full operator by iterating over basis states
    for s in range(dim):
        # Extract bits for q1 and q2
        b1 = (s >> (n_qubits - 1 - q1)) & 1
        b2 = (s >> (n_qubits - 1 - q2)) & 1
        # Only the {01, 10} subspace mixes
        if (b1, b2) in [(0, 1), (1, 0)]:
            # Find the partner state (swap q1 and q2)
            s_swap = s ^ (1 << (n_qubits - 1 - q1)) ^ (1 << (n_qubits - 1 - q2))
            idx_in  = 1 if b1 == 0 else 2  # |01>=1, |10>=2 in 2-qubit subspace
            idx_out1 = 1; idx_out2 = 2
            # col s -> mix with col s_swap
            for s2 in range(dim):
                b1_2 = (s2 >> (n_qubits - 1 - q1)) & 1
                b2_2 = (s2 >> (n_qubits - 1 - q2)) & 1
                if b1_2 == b1 and b2_2 == b2:
                    U[s, s2] = Uxy[idx_in, idx_in]
                elif s2 == s_swap:
                    idx_partner = 2 if idx_in == 1 else 1
                    U[s, s2] = Uxy[idx_in, idx_partner]
    return U


def build_ansatz_unitary(theta, n_qubits=4):
    """
    Build the full ansatz unitary from paper Eq.(9) as a matrix.

    For 4 qubits (NF=4):
      U(theta) = [Rz_0 Rz_1 Rz_2 Rz_3] * Uxy_12(theta_1) * Uxy_23(theta_2) * Uxy_01(theta_0)

    theta[0] = Uxy_01 angle
    theta[1] = Uxy_12 angle
    theta[2] = Uxy_23 angle
    theta[3..6] = Rz angles for qubits 0,1,2,3

    Application order: rightmost applied first (standard quantum convention).
    """
    dim = 2 ** n_qubits
    U = np.eye(dim, dtype=complex)

    # Step 1: Apply Uxy_01 and Uxy_23 (can be done in parallel, no overlap)
    U01 = np.kron(Uxy_matrix(theta[0]), np.eye(4, dtype=complex))
    U23 = np.kron(np.eye(4, dtype=complex), Uxy_matrix(theta[2]))
    U = U01 @ U
    U = U23 @ U

    # Step 2: Apply Uxy_12 (cross-flavor gate, creates entanglement between sectors)
    U12 = np.kron(np.kron(I2, Uxy_matrix(theta[1])), I2)
    U = U12 @ U

    # Step 3: Apply Rz on all qubits
    Rz_all = np.kron(
        np.kron(np.kron(Rz_matrix(theta[3]), Rz_matrix(theta[4])),
                Rz_matrix(theta[5])),
        Rz_matrix(theta[6])
    )
    U = Rz_all @ U

    return U


def apply_ansatz(theta, psi0):
    """Apply the ansatz unitary to the initial state."""
    U = build_ansatz_unitary(theta, n_qubits=int(np.log2(len(psi0))))
    return U @ psi0


def energy_statevector(theta, H, psi0):
    """
    Cost function for noiseless (statevector) VQE.
    <psi(theta)|H|psi(theta)> computed exactly.
    """
    psi = apply_ansatz(theta, psi0)
    return float(np.real(psi.conj() @ H @ psi))


# ============================================================================
# PART 3: SHOT-NOISE VQE
# This is what separates your work from pure linear algebra.
# A real quantum computer can only measure Pauli strings, one shot at a time.
# ============================================================================

def decompose_hamiltonian_pauli(H, n_qubits):
    """
    Decompose H into a sum of Pauli strings: H = sum_i c_i * P_i
    where each P_i is a tensor product of {I, X, Y, Z}.

    This is done by exploiting Tr[P_i * H] = 2^n * c_i for Pauli operators.
    Every 2^n x 2^n Hermitian matrix can be written this way.

    Why do we need this?
    A quantum computer can't measure H directly. It measures individual
    Pauli strings and combines the results. Each Pauli string measurement
    takes 'n_shots' circuit executions.
    """
    n = n_qubits
    dim = 2 ** n
    pauli_labels = []
    pauli_matrices = []
    pauli_coeffs = []

    paulis_1q = {'I': I2, 'X': X2, 'Y': Y2, 'Z': Z2}

    # Generate all 4^n Pauli strings
    def gen_strings(n):
        if n == 1:
            return list('IXYZ')
        return [s + c for s in gen_strings(n-1) for c in 'IXYZ']

    for label in gen_strings(n):
        # Build the tensor product matrix
        ops = [paulis_1q[c] for c in label]
        P = kron_list(ops)
        # Coefficient: c = Tr[P H] / 2^n
        c = np.trace(P.conj().T @ H) / dim
        if abs(c) > 1e-12:  # Skip negligible terms
            pauli_labels.append(label)
            pauli_matrices.append(P)
            pauli_coeffs.append(float(np.real(c)))

    return pauli_labels, pauli_matrices, pauli_coeffs


def measure_pauli_string(psi, pauli_matrix, n_shots):
    """
    Simulate measuring a Pauli string with finite shots.

    A Pauli operator P has eigenvalues ±1.
    The expectation value <P> = p(+1) - p(-1) where p(+1), p(-1) are
    probabilities of getting +1 or -1 eigenvalue.

    With n_shots measurements, we get a NOISY estimate:
    <P>_noisy = (n_plus - n_minus) / n_shots

    The variance of this estimate is:
    Var[<P>_noisy] = (1 - <P>^2) / n_shots

    This shot noise is the fundamental limit of NISQ VQE performance.
    The paper uses 100 shots per Pauli string -- we'll use the same.
    """
    # True expectation value
    true_exp = float(np.real(psi.conj() @ pauli_matrix @ psi))

    # Probability of getting eigenvalue +1
    p_plus = (1 + true_exp) / 2
    p_plus = np.clip(p_plus, 0, 1)  # numerical safety

    # Simulate n_shots binary outcomes: +1 with prob p_plus, -1 with prob 1-p_plus
    outcomes = np.random.choice([1, -1], size=n_shots,
                                p=[p_plus, 1 - p_plus])
    noisy_exp = np.mean(outcomes)
    return noisy_exp


def energy_shot_noise(theta, H_paulis, psi0, n_shots=100):
    """
    Cost function for shot-noise VQE.
    Estimates <H> by measuring each Pauli string with n_shots shots.

    This is what a real quantum computer does, except here we
    know the exact state psi(theta) -- we're just adding realistic noise
    to the measurement process.
    """
    psi = apply_ansatz(theta, psi0)
    labels, matrices, coeffs = H_paulis
    energy = 0.0
    for P, c in zip(matrices, coeffs):
        noisy_exp = measure_pauli_string(psi, P, n_shots)
        energy += c * noisy_exp
    return energy


# ============================================================================
# PART 4: ENTANGLEMENT ENTROPY
# This is your NEW physics contribution -- not in the base paper.
# ============================================================================

def von_neumann_entropy(rho):
    """
    S(rho) = -Tr[rho * log(rho)]

    For a pure state |psi>, the reduced density matrix rho_A = Tr_B[|psi><psi|]
    is mixed (S > 0) if and only if A and B are entangled.

    Physical meaning:
    - S = 0: subsystem A is in a pure state, no entanglement with B
    - S = log(2) ≈ 0.69: maximally entangled pair (Bell state)
    - S grows with entanglement

    For the Schwinger model phases:
    - Phase I (K<<0): ground state ≈ |0101> (product state) -> S ≈ 0
    - Phase II (K=0): ground state is entangled -> S > 0
    - Phase III (K>>0): ground state ≈ |1010> (product state) -> S ≈ 0
    This directly measures the ENTANGLEMENT STRUCTURE across the phase diagram.
    """
    eigenvalues = np.linalg.eigvalsh(rho)
    eigenvalues = eigenvalues[eigenvalues > 1e-15]  # remove numerical zeros
    return float(-np.sum(eigenvalues * np.log(eigenvalues)))


def entanglement_entropy(psi, partition_A, n_qubits):
    """
    Compute the von Neumann entropy of subsystem A (qubit indices in partition_A).

    Steps:
    1. Reshape psi into a tensor with one index per qubit
    2. Trace out subsystem B to get the reduced density matrix rho_A
    3. Compute S = -Tr[rho_A log rho_A]

    partition_A: list of qubit indices in subsystem A (0=MSB convention)
    """
    nA = len(partition_A)
    nB = n_qubits - nA
    partition_B = [q for q in range(n_qubits) if q not in partition_A]

    # Reshape psi: index i corresponds to qubit i (0=MSB)
    psi_tensor = psi.reshape([2] * n_qubits)

    # Move A indices first, B indices last
    perm = partition_A + partition_B
    psi_tensor = np.transpose(psi_tensor, perm)

    # Reshape to (dimA, dimB) matrix
    dimA = 2 ** nA
    dimB = 2 ** nB
    psi_matrix = psi_tensor.reshape(dimA, dimB)

    # Reduced density matrix rho_A = psi_matrix @ psi_matrix^\dagger
    rho_A = psi_matrix @ psi_matrix.conj().T

    return von_neumann_entropy(rho_A)


def compute_all_entropies(psi, n_qubits=4):
    """
    Compute entanglement entropy for all bipartitions of 4 qubits.
    Returns a dictionary of partition -> entropy.

    For 4 qubits, the three 2|2 bipartitions are:
    - (0,1)|(2,3): site-based -- qubits of same site together
    - (0,2)|(1,3): flavor-based -- same flavor together
    - (0,3)|(1,2): mixed

    These directly correspond to the QMI partitions in the paper (Fig. 4).
    """
    partitions = {
        'site: (0,1)|(2,3)': [0, 1],
        'flavor: (0,2)|(1,3)': [0, 2],
        'mixed: (0,3)|(1,2)': [0, 3]
    }
    return {name: entanglement_entropy(psi, A, n_qubits)
            for name, A in partitions.items()}


# ============================================================================
# PART 5: MAIN SIMULATION RUNS
# ============================================================================

def run_vqe_statevector(H, psi0, n_restarts=15, seed=42):
    """
    Noiseless VQE using exact statevector simulation.
    This is what we built in Orange.
    """
    rng = np.random.default_rng(seed)
    best_E, best_theta = np.inf, None
    for _ in range(n_restarts):
        t0 = rng.uniform(-np.pi, np.pi, 7)
        res = minimize(energy_statevector, t0, args=(H, psi0),
                       method='COBYLA',
                       options={'maxiter': 5000, 'rhobeg': 0.5})
        if res.fun < best_E:
            best_E, best_theta = res.fun, res.x
    return best_E, best_theta


def run_vqe_shot_noise(H_paulis, psi0, n_shots=100, n_restarts=8, seed=99):
    """
    Shot-noise VQE using finite measurements per Pauli string.
    n_shots=100 matches the paper exactly.

    NOTE: This is slower than statevector VQE because:
    1. Each energy evaluation requires measuring len(paulis) strings
    2. Each string requires n_shots simulated measurements
    3. The noisy cost function makes optimization harder

    We use COBYLA here too, but in real hardware SPSA is preferred
    because it handles the inherent randomness of shot-noise better.
    """
    rng = np.random.default_rng(seed)
    best_E, best_theta = np.inf, None
    for _ in range(n_restarts):
        t0 = rng.uniform(-np.pi, np.pi, 7)
        res = minimize(energy_shot_noise, t0, args=(H_paulis, psi0, n_shots),
                       method='COBYLA',
                       options={'maxiter': 2000, 'rhobeg': 0.5})
        if res.fun < best_E:
            best_E, best_theta = res.fun, res.x
    return best_E, best_theta


def run_convergence_trace(H, psi0, E_exact, n_iters=120, shot_noise=False,
                          n_shots=100, H_paulis=None, seed=7):
    """
    Run VQE with gradient descent and record energy at each step.
    Used to generate the convergence plots (like Fig. 3 in the paper).

    Parameters
    ----------
    shot_noise : bool
        If True, use shot-noise energy evaluation (like real hardware).
        If False, use exact statevector (like our classical simulation).
    """
    rng = np.random.default_rng(seed)
    theta = rng.uniform(-np.pi, np.pi, 7)
    eps = 1e-3

    trace = []
    trace_exact = []  # Track exact energy at each theta (even when using noisy evals)

    for it in range(n_iters):
        # Current energy (noisy or exact for plotting)
        if shot_noise and H_paulis is not None:
            E_current = energy_shot_noise(theta, H_paulis, psi0, n_shots)
        else:
            E_current = energy_statevector(theta, H, psi0)

        # Always track the TRUE energy at current parameters
        E_true = energy_statevector(theta, H, psi0)
        trace.append(E_current)
        trace_exact.append(E_true)

        # Gradient estimate
        grad = np.zeros(7)
        for j in range(7):
            tp = theta.copy(); tp[j] += eps
            tm = theta.copy(); tm[j] -= eps
            if shot_noise and H_paulis is not None:
                grad[j] = (energy_shot_noise(tp, H_paulis, psi0, n_shots) -
                           energy_shot_noise(tm, H_paulis, psi0, n_shots)) / (2 * eps)
            else:
                grad[j] = (energy_statevector(tp, H, psi0) -
                           energy_statevector(tm, H, psi0)) / (2 * eps)

        lr = 0.3 / (1 + 0.04 * it)
        theta -= lr * grad + rng.standard_normal(7) * 0.008

    return trace, trace_exact


# ============================================================================
# PART 6: SCALING STUDY (connects Orange to Jackfruit)
# ============================================================================

def scaling_study(x=16.0, N_max=5):
    """
    Systematic VQE scaling from N=2 to N_max lattice sites.

    For each N:
    - Build the Hamiltonian (grows exponentially with N)
    - Find exact ground state in Q_tot=0 sector
    - Run VQE with single-layer ansatz (fixed 7 parameters for all N)
    - Measure energy error and convergence

    KEY INSIGHT: For N=2, single-layer ansatz is COMPLETE (exact).
    For N>=3, it's INCOMPLETE (too few parameters for the larger subspace).
    The energy error starts growing -- this is the ansatz expressibility limit.

    Q_tot=0 sector dimensions:
    N=2: C(4,2) = 6   (7 params > 6 dim -> overcomplete, exact)
    N=3: C(6,3) = 20  (7 params < 20 dim -> undercomplete, approximate)
    N=4: C(8,4) = 70  (7 params << 70 dim -> very undercomplete)
    N=5: C(10,5)= 252 (7 params negligible)
    """
    results = []
    rng = np.random.default_rng(42)

    for N in range(2, N_max + 1):
        F = 2
        n_qubits = N * F
        dim = 2 ** n_qubits
        basis = get_qtot0_basis(n_qubits, F, N)
        Q0_dim = len(basis)

        print(f"  N={N}: {n_qubits} qubits, full dim={dim}, Q0 dim={Q0_dim}")

        H = build_hamiltonian(x, K=0.0, N=N, F=F)
        vals, psi_gs = exact_diagonalize(H, basis)
        E_exact = float(vals[0])

        # For VQE: use the paper's initial state generalized to N sites
        # |0101...01> (alternating 0 and 1 for each flavor pair)
        psi0 = np.zeros(dim, dtype=complex)
        init_bits = []
        for n in range(N):
            for f in range(F):
                init_bits.append(0 if f == 0 else 1)  # |0101...> pattern
        init_idx = sum(b * (2 ** (n_qubits - 1 - i))
                      for i, b in enumerate(init_bits))
        psi0[init_idx] = 1.0

        # VQE (single-layer, 7 params regardless of N)
        best_E = np.inf
        for restart in range(10):
            t0 = rng.uniform(-np.pi, np.pi, 7)
            res = minimize(energy_statevector, t0, args=(H, psi0),
                          method='COBYLA',
                          options={'maxiter': 3000, 'rhobeg': 0.5})
            if res.fun < best_E:
                best_E = res.fun
                best_theta = res.x

        # Entanglement entropy of exact ground state
        psi_gs_full = psi_gs
        if n_qubits == 4:
            entropies = compute_all_entropies(psi_gs_full, n_qubits)
        else:
            entropies = {}

        rel_err = abs(best_E - E_exact) / abs(E_exact) * 100 if E_exact != 0 else 0

        results.append({
            'N': N, 'n_qubits': n_qubits, 'Q0_dim': Q0_dim,
            'E_exact': E_exact, 'E_per_site': E_exact / N,
            'E_vqe': best_E, 'rel_err_pct': rel_err,
            'ansatz_params': 7,
            'entropies': entropies
        })

    return results


# ============================================================================
# PART 7: THE FULL STUDY -- put everything together
# ============================================================================

def main():
    print("=" * 65)
    print("SCHWINGER MODEL: STATEVECTOR vs SHOT-NOISE VQE COMPARISON")
    print("=" * 65)

    x = 16.0
    K_values = [-14.0, 0.0, 10.0]
    paper_exact = {-14.0: -223.0, 0.0: -30.7, 10.0: 1.0}
    paper_hardware = {-14.0: -215.8, 0.0: -26.6, 10.0: 2.5}

    # Initial state |0101>
    psi0 = np.zeros(16, dtype=complex)
    psi0[0b0101] = 1.0
    basis_4q = get_qtot0_basis(4)

    # Pre-decompose Hamiltonians into Pauli strings (needed for shot-noise VQE)
    print("\nDecomposing Hamiltonians into Pauli strings...")
    H_paulis_dict = {}
    for K in K_values:
        H = build_hamiltonian(x, K)
        labels, matrices, coeffs = decompose_hamiltonian_pauli(H, 4)
        H_paulis_dict[K] = (labels, matrices, coeffs)
        print(f"  K={K:>5}: {len(labels)} Pauli strings")

    # ---- STUDY 1: Statevector VQE ----
    print("\n[Study 1] Statevector VQE (noiseless)...")
    sv_results = {}
    for K in K_values:
        H = build_hamiltonian(x, K)
        vals, _ = exact_diagonalize(H, basis_4q)
        E_exact = float(vals[0])
        E_vqe, theta_opt = run_vqe_statevector(H, psi0, seed=42)
        psi_opt = apply_ansatz(theta_opt, psi0)
        N0 = float(np.real(psi_opt.conj() @ particle_number_op(0) @ psi_opt))
        N1 = float(np.real(psi_opt.conj() @ particle_number_op(1) @ psi_opt))
        entropies = compute_all_entropies(psi_opt, 4)
        sv_results[K] = {
            'E_exact': E_exact, 'E_vqe': E_vqe,
            'rel_err': abs(E_vqe - E_exact) / abs(E_exact) * 100,
            'N0': N0, 'N1': N1, 'entropies': entropies,
            'theta': theta_opt
        }
        print(f"  K={K:>5}: exact={E_exact:.4f}, VQE={E_vqe:.4f}, "
              f"err={abs(E_vqe-E_exact)/abs(E_exact)*100:.4f}%, "
              f"N0={N0:.2f}, N1={N1:.2f}")

    # ---- STUDY 2: Shot-noise VQE (100 shots per Pauli, like the paper) ----
    print("\n[Study 2] Shot-noise VQE (100 shots/Pauli, mimicking hardware)...")
    sn_results = {}
    np.random.seed(2025)
    for K in K_values:
        H = build_hamiltonian(x, K)
        E_exact = sv_results[K]['E_exact']
        E_vqe_sn, _ = run_vqe_shot_noise(H_paulis_dict[K], psi0,
                                          n_shots=100, n_restarts=8, seed=99)
        sn_results[K] = {
            'E_exact': E_exact, 'E_vqe': E_vqe_sn,
            'rel_err': abs(E_vqe_sn - E_exact) / abs(E_exact) * 100
        }
        print(f"  K={K:>5}: exact={E_exact:.4f}, "
              f"shot-noise VQE={E_vqe_sn:.4f}, "
              f"err={abs(E_vqe_sn-E_exact)/abs(E_exact)*100:.3f}%")

    # ---- STUDY 3: Convergence traces (statevector vs shot-noise) ----
    print("\n[Study 3] Convergence traces for K=0...")
    H_K0 = build_hamiltonian(x, 0.0)
    E_exact_K0 = sv_results[0.0]['E_exact']
    trace_sv, trace_sv_true = run_convergence_trace(
        H_K0, psi0, E_exact_K0, n_iters=100, shot_noise=False, seed=7)
    trace_sn, trace_sn_true = run_convergence_trace(
        H_K0, psi0, E_exact_K0, n_iters=100, shot_noise=True,
        n_shots=100, H_paulis=H_paulis_dict[0.0], seed=7)
    print(f"  Statevector final: {trace_sv_true[-1]:.4f} "
          f"(exact: {E_exact_K0:.4f})")
    print(f"  Shot-noise final:  {trace_sn_true[-1]:.4f} "
          f"(exact: {E_exact_K0:.4f})")

    # ---- STUDY 4: Phase diagram ----
    print("\n[Study 4] Full phase diagram...")
    K_sweep = np.linspace(-16, 16, 120)
    sectors_data = {(2,0):[], (1,1):[], (0,2):[]}
    entropy_sweep = []
    N0_sweep = []

    for K in K_sweep:
        H = build_hamiltonian(x, float(K))
        vals, psi_gs = exact_diagonalize(H, basis_4q)
        # Per-sector energies
        for (n0, n1), idx_list in [
            ((2,0), [5]), ((1,1), [3,6,9,12]), ((0,2), [10])
        ]:
            Hsub = np.array([[H[i,j] for j in idx_list] for i in idx_list])
            sectors_data[(n0,n1)].append(float(eigh(Hsub)[0][0]))
        # Entanglement entropy of global ground state (bipartition 0,1|2,3)
        S = entanglement_entropy(psi_gs, [0, 1], 4)
        entropy_sweep.append(S)
        # <N0>
        N0_op = particle_number_op(0)
        N0_sweep.append(float(np.real(psi_gs.conj() @ N0_op @ psi_gs)))

    # Phase boundaries
    E20 = np.array(sectors_data[(2,0)])
    E11 = np.array(sectors_data[(1,1)])
    E02 = np.array(sectors_data[(0,2)])
    K_crits = []
    for i in range(1, len(K_sweep)):
        if (E20[i-1]-E11[i-1]) * (E20[i]-E11[i]) < 0:
            Kc = K_sweep[i-1] + (K_sweep[i]-K_sweep[i-1]) * \
                 (E20[i-1]-E11[i-1]) / (E20[i-1]-E11[i-1]-E20[i]+E11[i])
            K_crits.append(Kc)
        if (E11[i-1]-E02[i-1]) * (E11[i]-E02[i]) < 0:
            Kc = K_sweep[i-1] + (K_sweep[i]-K_sweep[i-1]) * \
                 (E11[i-1]-E02[i-1]) / (E11[i-1]-E02[i-1]-E11[i]+E02[i])
            K_crits.append(Kc)
    print(f"  Phase boundaries: {[f'{k:.4f}' for k in sorted(K_crits)]}")
    print(f"  Paper exact: ±3.96")

    # ---- STUDY 5: Scaling ----
    print("\n[Study 5] VQE scaling N=2 to 5...")
    scale_results = scaling_study(x=x, N_max=5)
    print("\n  N | nq | Q0_dim | E_exact/site | E_VQE/site | Rel.err%")
    print("  " + "-"*58)
    for r in scale_results:
        vqe_per_site = r['E_vqe'] / r['N'] if r['E_vqe'] < np.inf else float('nan')
        print(f"  {r['N']} | {r['n_qubits']:2d} | {r['Q0_dim']:6d} | "
              f"{r['E_per_site']:12.4f} | {vqe_per_site:10.4f} | "
              f"{r['rel_err_pct']:8.3f}%")

    # ---- GENERATE ALL FIGURES ----
    print("\n[Generating figures...]")
    generate_all_figures(
        K_values, sv_results, sn_results, paper_exact, paper_hardware,
        K_sweep, E20, E11, E02, K_crits, entropy_sweep, N0_sweep,
        trace_sv, trace_sn, trace_sv_true, trace_sn_true, E_exact_K0,
        scale_results
    )

    print("\nDone. Figures saved:")
    for i in range(1, 6):
        print(f"  fig{i}_*.png")


# ============================================================================
# PART 8: FIGURES
# ============================================================================

def generate_all_figures(K_values, sv_results, sn_results,
                         paper_exact, paper_hardware,
                         K_sweep, E20, E11, E02, K_crits,
                         entropy_sweep, N0_sweep,
                         trace_sv, trace_sn, trace_sv_true, trace_sn_true,
                         E_exact_K0, scale_results):

    plt.rcParams.update({'font.family': 'serif', 'font.size': 11,
                         'axes.labelsize': 12, 'figure.facecolor': 'white'})
    colors = {-14.0: '#1f77b4', 0.0: '#d62728', 10.0: '#2ca02c'}
    markers = {-14.0: 'o', 0.0: 's', 10.0: '^'}

    # ---- Figure 1: Phase diagram with entanglement entropy ----
    fig = plt.figure(figsize=(14, 5))
    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])

    # Panel a: Sector energies
    ax1.plot(K_sweep, E20, 'b-',  lw=2, label=r'$\mathcal{N}=(2,0)$')
    ax1.plot(K_sweep, E11, 'r-',  lw=2, label=r'$\mathcal{N}=(1,1)$')
    ax1.plot(K_sweep, E02, 'g--', lw=2, label=r'$\mathcal{N}=(0,2)$')
    for Kc in K_crits:
        ax1.axvline(Kc, color='gray', ls=':', lw=1.5, alpha=0.7)
    for K in K_values:
        ax1.scatter([K], [sv_results[K]['E_vqe']], color=colors[K],
                    s=100, marker=markers[K], zorder=5, edgecolors='k', lw=0.8,
                    label=f'VQE K={K:+.0f}')
    ax1.axvspan(-16, K_crits[0], alpha=0.05, color='blue')
    ax1.axvspan(K_crits[0], K_crits[1], alpha=0.05, color='red')
    ax1.axvspan(K_crits[1], 16, alpha=0.05, color='green')
    ax1.set_xlabel(r'$K = \kappa_0/g$')
    ax1.set_ylabel(r'$\langle W \rangle$')
    ax1.set_title('(a) Phase diagram', fontsize=11)
    ax1.legend(fontsize=8, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-16, 16)

    # Panel b: N0 expectation
    ax2.plot(K_sweep, N0_sweep, 'k-', lw=2.5)
    for Kc in K_crits:
        ax2.axvline(Kc, color='gray', ls=':', lw=1.5, alpha=0.7)
    for K in K_values:
        ax2.scatter([K], [sv_results[K]['N0']], color=colors[K],
                    s=100, marker=markers[K], zorder=5, edgecolors='k', lw=0.8)
    ax2.set_xlabel(r'$K = \kappa_0/g$')
    ax2.set_ylabel(r'$\langle N_0 \rangle$')
    ax2.set_title(r'(b) Flavor-0 particle number', fontsize=11)
    ax2.set_ylim(-0.2, 2.4)
    ax2.axvspan(-16, K_crits[0], alpha=0.05, color='blue')
    ax2.axvspan(K_crits[0], K_crits[1], alpha=0.05, color='red')
    ax2.axvspan(K_crits[1], 16, alpha=0.05, color='green')
    ax2.grid(True, alpha=0.3)

    # Panel c: Entanglement entropy -- NEW CONTRIBUTION
    ax3.plot(K_sweep, entropy_sweep, 'purple', lw=2.5)
    for Kc in K_crits:
        ax3.axvline(Kc, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax3.axhline(0, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax3.set_xlabel(r'$K = \kappa_0/g$')
    ax3.set_ylabel(r'$S_{(0,1)|(2,3)}$ (nats)')
    ax3.set_title('(c) Entanglement entropy [NEW]', fontsize=11)
    ax3.axvspan(-16, K_crits[0], alpha=0.05, color='blue')
    ax3.axvspan(K_crits[0], K_crits[1], alpha=0.05, color='red')
    ax3.axvspan(K_crits[1], 16, alpha=0.05, color='green')
    ax3.grid(True, alpha=0.3)
    # Annotate phases
    ax3.text(-10, max(entropy_sweep)*0.5, 'Phase I\nS≈0', ha='center',
             fontsize=9, color='#1f77b4')
    ax3.text(0, max(entropy_sweep)*0.9, 'Phase II\nS>0', ha='center',
             fontsize=9, color='#d62728')
    ax3.text(10, max(entropy_sweep)*0.5, 'Phase III\nS≈0', ha='center',
             fontsize=9, color='#2ca02c')

    plt.suptitle('Phase Diagram, Particle Number, and Entanglement Entropy\n'
                 'Two-flavor Schwinger model: N=2, F=2, x=16', fontsize=11)
    plt.savefig('fig1_phase_diagram_entropy.png', dpi=200, bbox_inches='tight')
    plt.close()
    print("  fig1_phase_diagram_entropy.png")

    # ---- Figure 2: Statevector vs Shot-noise convergence (K=0) ----
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    iters = range(len(trace_sv))
    # Left: measured energy (noisy vs exact)
    ax1.plot(iters, trace_sv, 'b-', lw=1.5, alpha=0.9,
             label='Statevector (noiseless)')
    ax1.plot(iters, trace_sn, 'r-', lw=1, alpha=0.7,
             label='Shot-noise (100 shots/Pauli)')
    ax1.axhline(E_exact_K0, color='k', ls='--', lw=1.5,
                label=f'Exact: {E_exact_K0:.3f}')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel(r'$\langle W(\theta) \rangle$ (measured)')
    ax1.set_title('(a) Measured cost function during VQE, K=0', fontsize=11)
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    # Right: true energy at current theta (both use this)
    ax2.plot(iters, trace_sv_true, 'b-', lw=2, alpha=0.9,
             label='Statevector VQE')
    ax2.plot(iters, trace_sn_true, 'r-', lw=1.5, alpha=0.8,
             label='Shot-noise VQE')
    ax2.axhline(E_exact_K0, color='k', ls='--', lw=1.5,
                label=f'Exact: {E_exact_K0:.3f}')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel(r'True $\langle W(\theta) \rangle$ (noiseless eval)')
    ax2.set_title('(b) True energy at current parameters, K=0', fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Statevector vs Shot-noise VQE Convergence\n'
                 'K=0 (Phase II), 100 shots per Pauli string', fontsize=11)
    plt.tight_layout()
    plt.savefig('fig2_sv_vs_shotnoise.png', dpi=200, bbox_inches='tight')
    plt.close()
    print("  fig2_sv_vs_shotnoise.png")

    # ---- Figure 3: Three-way energy comparison ----
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    K_labels = ['K=-14', 'K=0', 'K=+10']
    K_keys = K_values
    x_pos = np.arange(3)
    w = 0.22

    E_ex   = [sv_results[k]['E_exact']   for k in K_keys]
    E_sv   = [sv_results[k]['E_vqe']     for k in K_keys]
    E_sn   = [sn_results[k]['E_vqe']     for k in K_keys]
    E_hw   = [paper_hardware[k]          for k in K_keys]

    ax = axes[0]
    b1 = ax.bar(x_pos - 1.5*w, np.abs(E_ex), w, label='Exact ED',
                color='#333333', edgecolor='k', lw=0.5)
    b2 = ax.bar(x_pos - 0.5*w, np.abs(E_sv), w, label='Statevector VQE',
                color='steelblue', edgecolor='k', lw=0.5)
    b3 = ax.bar(x_pos + 0.5*w, np.abs(E_sn), w, label='Shot-noise VQE (100)',
                color='salmon', edgecolor='k', lw=0.5)
    b4 = ax.bar(x_pos + 1.5*w, np.abs(E_hw), w, label='Hardware (Melzer+25)',
                color='gold', edgecolor='k', lw=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(K_labels, fontsize=11)
    ax.set_ylabel(r'$|\langle W \rangle|$')
    ax.set_title('(a) Energy comparison: all methods', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, axis='y', alpha=0.3)

    # Relative errors
    ax2 = axes[1]
    errs_sv = [sv_results[k]['rel_err']  for k in K_keys]
    errs_sn = [sn_results[k]['rel_err']  for k in K_keys]
    errs_hw = [abs(paper_hardware[k]-paper_exact[k])/abs(paper_exact[k])*100
               for k in K_keys]

    ax2.bar(x_pos - w, errs_sv, w, label='Statevector VQE',
            color='steelblue', edgecolor='k', lw=0.5)
    ax2.bar(x_pos,     errs_sn, w, label='Shot-noise VQE',
            color='salmon', edgecolor='k', lw=0.5)
    ax2.bar(x_pos + w, errs_hw, w, label='Hardware (Melzer+25)',
            color='gold', edgecolor='k', lw=0.5)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(K_labels, fontsize=11)
    ax2.set_ylabel('Relative error (%)')
    ax2.set_title('(b) Relative error vs exact diagonalization', fontsize=11)
    ax2.legend(fontsize=8)
    ax2.grid(True, axis='y', alpha=0.3)
    ax2.set_yscale('log')

    plt.suptitle('Three-way Comparison: Statevector / Shot-noise / Hardware',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig('fig3_threeway_comparison.png', dpi=200, bbox_inches='tight')
    plt.close()
    print("  fig3_threeway_comparison.png")

    # ---- Figure 4: Entanglement entropy per phase ----
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Per-phase entropy bar chart
    ax = axes[0]
    partition_names = ['site\n(0,1)|(2,3)', 'flavor\n(0,2)|(1,3)',
                       'mixed\n(0,3)|(1,2)']
    entropy_keys = list(sv_results[K_values[0]]['entropies'].keys())
    x_pos = np.arange(3)
    width = 0.25
    phase_colors = ['#1f77b4', '#d62728', '#2ca02c']
    phase_labels = ['Phase I (K=-14)', 'Phase II (K=0)', 'Phase III (K=+10)']

    for i, K in enumerate(K_values):
        vals = [sv_results[K]['entropies'].get(k, 0) for k in entropy_keys]
        ax.bar(x_pos + (i-1)*width, vals, width,
               label=phase_labels[i], color=phase_colors[i],
               edgecolor='k', lw=0.5, alpha=0.85)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(partition_names, fontsize=9)
    ax.set_ylabel('Von Neumann entropy S (nats)')
    ax.set_title('(a) Entanglement entropy by bipartition [NEW]', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)

    # Entropy vs K for all bipartitions
    ax2 = axes[1]
    # Recompute for all K
    K_fine = np.linspace(-16, 16, 80)
    basis = get_qtot0_basis(4)
    H_vals = [build_hamiltonian(16.0, float(K)) for K in K_fine]
    psi_vals = [exact_diagonalize(H, basis)[1] for H in H_vals]
    part_labels_short = ['site (0,1)|(2,3)', 'flavor (0,2)|(1,3)',
                         'mixed (0,3)|(1,2)']
    part_colors = ['purple', 'teal', 'orange']
    partitions_A = [[0,1], [0,2], [0,3]]
    for pA, lbl, col in zip(partitions_A, part_labels_short, part_colors):
        S_vals = [entanglement_entropy(psi, pA, 4) for psi in psi_vals]
        ax2.plot(K_fine, S_vals, color=col, lw=2, label=lbl)
    for Kc in K_crits:
        ax2.axvline(Kc, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax2.set_xlabel(r'$K = \kappa_0/g$')
    ax2.set_ylabel('S (nats)')
    ax2.set_title('(b) Entanglement entropy vs K (all bipartitions)', fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Entanglement Structure of the Schwinger Model Ground State',
                 fontsize=11)
    plt.tight_layout()
    plt.savefig('fig4_entanglement.png', dpi=200, bbox_inches='tight')
    plt.close()
    print("  fig4_entanglement.png")

    # ---- Figure 5: Scaling study ----
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))

    N_vals = [r['N'] for r in scale_results]
    Q0_dims = [r['Q0_dim'] for r in scale_results]
    rel_errs = [r['rel_err_pct'] for r in scale_results]
    E_per_site = [r['E_per_site'] for r in scale_results]

    # Panel a: Q0 sector dimension vs N
    ax = axes[0]
    ax.bar(N_vals, Q0_dims, color='steelblue', edgecolor='k', lw=0.5, alpha=0.85)
    ax.axhline(7, color='red', ls='--', lw=1.5,
               label='Ansatz params = 7')
    for N, d in zip(N_vals, Q0_dims):
        ax.text(N, d + 2, str(d), ha='center', va='bottom', fontsize=11)
    ax.set_xlabel('Lattice sites N')
    ax.set_ylabel(r'$Q_\mathrm{tot}=0$ sector dimension')
    ax.set_title('(a) Hilbert space growth', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    ax.set_xticks(N_vals)

    # Panel b: VQE relative error vs N
    ax = axes[1]
    mask_valid = [r['rel_err_pct'] < 100 for r in scale_results]
    N_valid = [n for n, v in zip(N_vals, mask_valid) if v]
    err_valid = [e for e, v in zip(rel_errs, mask_valid) if v]
    ax.semilogy(N_valid, [max(e, 1e-4) for e in err_valid], 'ro-',
                lw=2, ms=8, markeredgecolor='k', lw2=0.8)
    ax.axvline(2.5, color='gray', ls=':', lw=1.5, alpha=0.7)
    ax.text(2.1, max(err_valid)*0.5, 'Ansatz\ncomplete', ha='right',
            fontsize=9, color='gray')
    ax.text(2.6, max(err_valid)*0.5, 'Ansatz\nincomplete', ha='left',
            fontsize=9, color='red')
    ax.set_xlabel('Lattice sites N')
    ax.set_ylabel('VQE relative error (%)')
    ax.set_title('(b) VQE accuracy vs system size', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(N_vals)

    # Panel c: Ground state energy per site (finite-size scaling)
    ax = axes[2]
    ax.plot(N_vals, E_per_site, 'bo-', lw=2, ms=8, markeredgecolor='k', lw2=0.8)
    for N, E in zip(N_vals, E_per_site):
        ax.text(N, E - 0.5, f'{E:.2f}', ha='center', va='top', fontsize=9)
    ax.set_xlabel('Lattice sites N')
    ax.set_ylabel(r'$\langle W \rangle / N$ (energy per site)')
    ax.set_title('(c) Finite-size energy scaling (K=0)', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(N_vals)

    plt.suptitle('VQE Scaling Study: N=2 to 5 Lattice Sites\n'
                 'x=16, K=0, Single-layer ansatz (7 parameters)', fontsize=11)
    plt.tight_layout()
    plt.savefig('fig5_scaling.png', dpi=200, bbox_inches='tight')
    plt.close()
    print("  fig5_scaling.png")


# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    main()

SCHWINGER MODEL: STATEVECTOR vs SHOT-NOISE VQE COMPARISON

Decomposing Hamiltonians into Pauli strings...
  K=-14.0: 9 Pauli strings
  K=  0.0: 8 Pauli strings
  K= 10.0: 9 Pauli strings

[Study 1] Statevector VQE (noiseless)...
  K=-14.0: exact=-223.0000, VQE=-223.0000, err=0.0000%, N0=2.00, N1=0.00
  K=  0.0: exact=-30.5644, VQE=-30.5606, err=0.0127%, N0=1.00, N1=1.00
  K= 10.0: exact=1.0000, VQE=1.0000, err=0.0000%, N0=0.00, N1=2.00

[Study 2] Shot-noise VQE (100 shots/Pauli, mimicking hardware)...
  K=-14.0: exact=-223.0000, shot-noise VQE=-222.5900, err=0.184%
  K=  0.0: exact=-30.5644, shot-noise VQE=-29.7600, err=2.632%
  K= 10.0: exact=1.0000, shot-noise VQE=4.5400, err=354.000%

[Study 3] Convergence traces for K=0...
  Statevector final: -29.1329 (exact: -30.5644)
  Shot-noise final:  -2.5803 (exact: -30.5644)

[Study 4] Full phase diagram...
  Phase boundaries: ['-3.9456', '3.9456']
  Paper exact: ±3.96

[Study 5] VQE scaling N=2 to 5...
  N=2: 4 qubits, full dim=16, Q0 dim=

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 64 is different from 16)